In [1]:
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms.functional as F
import numpy as np


DATASET_DIR = Path("/kaggle/input/datasets/emiliotg/proyectoia-camisasdefutbol/IA")

TRAIN_RATIO = 0.70
SEED = 42

BATCH_SIZE = 2
NUM_WORKERS = 0

CLASS_TO_IDX = {
    "argentina": 1,
    "boca": 2,
    "barca": 3,
    "man utd": 4,
    "mexico": 5,
}

IDX_TO_CLASS = {
    1: "argentina",
    2: "boca",
    3: "barca",
    4: "man utd",
    5: "mexico",
}

CARPETA_A_ETIQUETA = {
    "Argentina": "argentina",
    "Boca Juniors": "boca",
    "FC Barcelona": "barca",
    "Manchester United": "man utd",
    "Mexico": "mexico",
}

def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes = []
    labels = []

    for obj in root.findall("object"):
        name_tag = obj.find("name")

        if name_tag is None:
            raise ValueError(f"Objeto sin etiqueta <name> en {xml_path}")

        class_name = name_tag.text.strip()

        if class_name not in CLASS_TO_IDX:
            raise ValueError(f"Clase desconocida '{class_name}' en {xml_path}")

        bndbox = obj.find("bndbox")

        if bndbox is None:
            raise ValueError(f"Objeto sin <bndbox> en {xml_path}")

        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        if xmax <= xmin or ymax <= ymin:
            raise ValueError(f"Caja inválida en {xml_path}: {xmin, ymin, xmax, ymax}")

        boxes.append([xmin, ymin, xmax, ymax])
        labels.append(CLASS_TO_IDX[class_name])

    if len(boxes) == 0:
        raise ValueError(f"XML sin objetos: {xml_path}")

    boxes = torch.as_tensor(boxes, dtype=torch.float32)
    labels = torch.as_tensor(labels, dtype=torch.int64)

    return boxes, labels

class EquiposDataset(Dataset):
    def __init__(self, dataset_dir):
        self.dataset_dir = Path(dataset_dir)
        self.items = []

        xml_dir = self.dataset_dir / "Dibujo"
        xml_files = sorted(xml_dir.glob("*.xml"))

        total_xml = len(xml_files)
        ignored_no_jpg = 0
        ignored_invalid_xml = 0

        for carpeta, etiqueta in CARPETA_A_ETIQUETA.items():
            clase_dir = self.dataset_dir / carpeta
            if not clase_dir.exists():
                print(f" Carpeta no encontrada: {clase_dir}")
                continue

            for img_path in clase_dir.glob("*.jpg"):
                xml_path = xml_dir / f"{img_path.stem}.xml"
                
                if not xml_path.exists():
                    ignored_no_jpg += 1
                    continue

                try:
                    parse_voc_xml(xml_path)
                except Exception as e:
                    ignored_invalid_xml += 1
                    print(f"XML ignorado: {xml_path.name} | Error: {e}")
                    continue

                self.items.append((img_path, xml_path))

        print("Revisión del dataset terminada")
        print("-" * 60)
        print(f"XML encontrados: {total_xml}")
        print(f"Pares válidos imagen + XML: {len(self.items)}")
        print(f"XML ignorados porque no existe JPG: {ignored_no_jpg}")
        print(f"XML ignorados por error interno: {ignored_invalid_xml}")
        print("-" * 60)

        if len(self.items) == 0:
            raise RuntimeError("No se encontró ningún par válido imagen + XML.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, xml_path = self.items[idx]

        image = Image.open(img_path).convert("RGB")
        
        # Guardar tamaño original de la imagen
        original_width, original_height = image.size

        # Tamaño al que se va a reducir la imagen
        target_size = (640, 640)
        image = image.resize(target_size, Image.Resampling.LANCZOS)
        
        # Calcular cuánto se encogió la imagen 
        scale_x = target_size[0] / original_width
        scale_y = target_size[1] / original_height

        image = F.to_tensor(image)
        boxes, labels = parse_voc_xml(xml_path)
        
        # Hacer una copia de las cajas para no modificar las originales
        boxes_scaled = boxes.clone()
        
        # Hacer más chicas las coordenadas X, Y en la misma proporción que la imagen
        boxes_scaled[:, [0, 2]] *= scale_x
        boxes_scaled[:, [1, 3]] *= scale_y
        
        # Evitar que el borde del bounding box se salga de la imagen
        boxes_scaled[:, 0] = torch.clamp(boxes_scaled[:, 0], min=0, max=target_size[0])
        boxes_scaled[:, 2] = torch.clamp(boxes_scaled[:, 2], min=0, max=target_size[0])
        boxes_scaled[:, 1] = torch.clamp(boxes_scaled[:, 1], min=0, max=target_size[1])
        boxes_scaled[:, 3] = torch.clamp(boxes_scaled[:, 3], min=0, max=target_size[1])

        area = (boxes_scaled[:, 2] - boxes_scaled[:, 0]) * (boxes_scaled[:, 3] - boxes_scaled[:, 1])
        iscrowd = torch.zeros((boxes_scaled.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes_scaled,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd,
        }

        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = EquiposDataset(dataset_dir=DATASET_DIR)

indices = list(range(len(dataset)))
random.seed(SEED)
random.shuffle(indices)

train_size = int(len(indices) * TRAIN_RATIO)

train_indices = indices[:train_size]
valid_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
valid_dataset = Subset(dataset, valid_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print("Dataset cargado correctamente")
print("-" * 60)
print(f"Total pares válidos: {len(dataset)}")
print(f"Train: {len(train_dataset)} imágenes")
print(f"Valid: {len(valid_dataset)} imágenes")
print(f"Clases: {CLASS_TO_IDX}")
print("-" * 60)

Revisión del dataset terminada
------------------------------------------------------------
XML encontrados: 350
Pares válidos imagen + XML: 350
XML ignorados porque no existe JPG: 0
XML ignorados por error interno: 0
------------------------------------------------------------
Dataset cargado correctamente
------------------------------------------------------------
Total pares válidos: 350
Train: 244 imágenes
Valid: 106 imágenes
Clases: {'argentina': 1, 'boca': 2, 'barca': 3, 'man utd': 4, 'mexico': 5}
------------------------------------------------------------


In [2]:
import torch
import torchvision

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from tqdm.auto import tqdm

NUM_CLASSES = 6  # 5 clases + background
NUM_EPOCHS = 6

LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

IDX_TO_CLASS = {
    1: "argentina",
    2: "boca",
    3: "barca",
    4: "man utd",
    5: "mexico",
}

EVAL_CLASSES = [1, 2, 3, 4, 5]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(weights=weights)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

model.to(device)

params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)


def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()

    total_loss = 0.0

    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch} - Train")

    for images, targets in progress_bar:
        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        loss_dict = model(images, targets)

        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        loss_value = losses.item()
        total_loss += loss_value

        progress_bar.set_postfix(loss=loss_value)

    avg_loss = total_loss / len(data_loader)
    return avg_loss

@torch.no_grad()
def validate_one_epoch(model, data_loader, device, epoch):
   
    model.train()

    total_loss = 0.0

    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch} - Valid Loss")

    for images, targets in progress_bar:
        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        loss_dict = model(images, targets)

        losses = sum(loss for loss in loss_dict.values())

        loss_value = losses.item()
        total_loss += loss_value

        progress_bar.set_postfix(loss=loss_value)

    avg_loss = total_loss / len(data_loader)
    return avg_loss

def compute_ap_for_class(predictions, ground_truths, class_id, iou_threshold):
    """
    predictions:
        lista de dicts:
        {
            "image_id": int,
            "box": tensor [4],
            "score": float,
            "label": int
        }

    ground_truths:
        lista de dicts:
        {
            "image_id": int,
            "boxes": tensor [N, 4],
            "labels": tensor [N]
        }
    """
    gt_by_image = {}
    total_gt = 0

    for gt in ground_truths:
        image_id = gt["image_id"]

        boxes = gt["boxes"]
        labels = gt["labels"]

        mask = labels == class_id
        class_boxes = boxes[mask]

        if class_boxes.numel() > 0:
            gt_by_image[image_id] = {
                "boxes": class_boxes,
                "matched": torch.zeros((class_boxes.shape[0],), dtype=torch.bool)
            }
            total_gt += class_boxes.shape[0]

    if total_gt == 0:
        return None

    class_predictions = [
        pred for pred in predictions
        if pred["label"] == class_id
    ]

    if len(class_predictions) == 0:
        return 0.0

    class_predictions = sorted(
        class_predictions,
        key=lambda x: x["score"],
        reverse=True
    )

    tp = []
    fp = []

    for pred in class_predictions:
        image_id = pred["image_id"]
        pred_box = pred["box"].unsqueeze(0)

        if image_id not in gt_by_image:
            tp.append(0.0)
            fp.append(1.0)
            continue

        gt_boxes = gt_by_image[image_id]["boxes"]
        matched = gt_by_image[image_id]["matched"]

        ious = torchvision.ops.box_iou(pred_box, gt_boxes)[0]

        max_iou, max_idx = torch.max(ious, dim=0)

        if max_iou >= iou_threshold and not matched[max_idx]:
            tp.append(1.0)
            fp.append(0.0)
            matched[max_idx] = True
        else:
            tp.append(0.0)
            fp.append(1.0)

    tp = torch.tensor(tp, dtype=torch.float32)
    fp = torch.tensor(fp, dtype=torch.float32)

    cumulative_tp = torch.cumsum(tp, dim=0)
    cumulative_fp = torch.cumsum(fp, dim=0)

    recalls = cumulative_tp / total_gt
    precisions = cumulative_tp / torch.clamp(
        cumulative_tp + cumulative_fp,
        min=1e-6
    )

    recalls = torch.cat([
        torch.tensor([0.0]),
        recalls,
        torch.tensor([1.0])
    ])

    precisions = torch.cat([
        torch.tensor([0.0]),
        precisions,
        torch.tensor([0.0])
    ])

    for i in range(precisions.numel() - 2, -1, -1):
        precisions[i] = torch.maximum(precisions[i], precisions[i + 1])

    changing_points = torch.where(recalls[1:] != recalls[:-1])[0]

    ap = torch.sum(
        (recalls[changing_points + 1] - recalls[changing_points]) *
        precisions[changing_points + 1]
    )

    return float(ap.item())

@torch.no_grad()
def evaluate_map(model, data_loader, device, iou_threshold):
    model.eval()

    all_predictions = []
    all_ground_truths = []

    image_counter = 0

    progress_bar = tqdm(
        data_loader,
        desc=f"Evaluando mAP@{iou_threshold:.2f}"
    )

    for images, targets in progress_bar:
        images_gpu = [img.to(device) for img in images]

        outputs = model(images_gpu)

        for output, target in zip(outputs, targets):
            image_id = image_counter

            gt_boxes = target["boxes"].cpu()
            gt_labels = target["labels"].cpu()

            all_ground_truths.append({
                "image_id": image_id,
                "boxes": gt_boxes,
                "labels": gt_labels
            })

            pred_boxes = output["boxes"].detach().cpu()
            pred_labels = output["labels"].detach().cpu()
            pred_scores = output["scores"].detach().cpu()

            for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
                label_id = int(label.item())

                if label_id not in EVAL_CLASSES:
                    continue

                all_predictions.append({
                    "image_id": image_id,
                    "box": box,
                    "label": label_id,
                    "score": float(score.item())
                })

            image_counter += 1

    aps = []

    for class_id in EVAL_CLASSES:
        ap = compute_ap_for_class(
            predictions=all_predictions,
            ground_truths=all_ground_truths,
            class_id=class_id,
            iou_threshold=iou_threshold
        )

        if ap is not None:
            aps.append(ap)

    if len(aps) == 0:
        return 0.0

    map_value = sum(aps) / len(aps)

    return map_value

history = {
    "train_loss": [],
    "valid_loss": [],
    "map_50": [],
    "map_75": [],
    "map_avg": [],
}

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = train_one_epoch(
        model=model,
        optimizer=optimizer,
        data_loader=train_loader,
        device=device,
        epoch=epoch
    )

    valid_loss = validate_one_epoch(
        model=model,
        data_loader=valid_loader,
        device=device,
        epoch=epoch
    )

    map_50 = evaluate_map(
        model=model,
        data_loader=valid_loader,
        device=device,
        iou_threshold=0.50
    )

    map_75 = evaluate_map(
        model=model,
        data_loader=valid_loader,
        device=device,
        iou_threshold=0.75
    )

    map_avg = (map_50 + map_75) / 2

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["map_50"].append(map_50)
    history["map_75"].append(map_75)
    history["map_avg"].append(map_avg)

    print("-" * 70)
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"Train loss : {train_loss:.4f}")
    print(f"Valid loss : {valid_loss:.4f}")
    print(f"mAP@0.50  : {map_50:.4f}")
    print(f"mAP@0.75  : {map_75:.4f}")
    print(f"mAP avg   : {map_avg:.4f}")
    print("-" * 70)

avg_train_loss = sum(history["train_loss"]) / len(history["train_loss"])
avg_valid_loss = sum(history["valid_loss"]) / len(history["valid_loss"])
avg_map_50 = sum(history["map_50"]) / len(history["map_50"])
avg_map_75 = sum(history["map_75"]) / len(history["map_75"])
avg_map_total = sum(history["map_avg"]) / len(history["map_avg"])

print("\nEntrenamiento terminado")
print("=" * 70)
print("PROMEDIO FINAL DE TODAS LAS ÉPOCAS")
print("=" * 70)
print(f"Promedio Train loss : {avg_train_loss:.4f}")
print(f"Promedio Valid loss : {avg_valid_loss:.4f}")
print(f"Promedio mAP@0.50   : {avg_map_50:.4f}")
print(f"Promedio mAP@0.75   : {avg_map_75:.4f}")
print(f"Promedio mAP total  : {avg_map_total:.4f}")
print("=" * 70)

Device: cuda


Epoch 1 - Train:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 1 - Valid Loss:   0%|          | 0/53 [00:00<?, ?it/s]

Evaluando mAP@0.50:   0%|          | 0/53 [00:00<?, ?it/s]

Evaluando mAP@0.75:   0%|          | 0/53 [00:00<?, ?it/s]

...

Epoch 6 - Train:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 6 - Valid Loss:   0%|          | 0/53 [00:00<?, ?it/s]

Evaluando mAP@0.50:   0%|          | 0/53 [00:00<?, ?it/s]

Evaluando mAP@0.75:   0%|          | 0/53 [00:00<?, ?it/s]

----------------------------------------------------------------------
Epoch 6/6
Train loss : 0.0629
Valid loss : 0.0959
mAP@0.50  : 0.8912
mAP@0.75  : 0.8131
mAP avg   : 0.8521
----------------------------------------------------------------------

Entrenamiento terminado
PROMEDIO FINAL DE TODAS LAS ÉPOCAS
Promedio Train loss : 0.1471
Promedio Valid loss : 0.1498
Promedio mAP@0.50   : 0.7042
Promedio mAP@0.75   : 0.5195
Promedio mAP total  : 0.6119


In [ ]:
import cv2
import torch
import threading
import time
from collections import deque
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as F
from PIL import Image

Ruta_pesos = "Faster_R-CNN_camisasfutbol.pth"
Umbral_confianza = 0.50
Num_clases = 6

Indice_a_clase = {
    1: "argentina",
    2: "boca",
    3: "barca",
    4: "man utd",
    5: "mexico",
}

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Iniciando modelo con: {dispositivo}")

print("Cargando el cerebro de tu modelo.")
modelo = fasterrcnn_resnet50_fpn(weights=None, min_size=600, max_size=800)
caracteristicas_entrada = modelo.roi_heads.box_predictor.cls_score.in_features
modelo.roi_heads.box_predictor = FastRCNNPredictor(caracteristicas_entrada, Num_clases)
modelo.load_state_dict(torch.load(Ruta_pesos, map_location=dispositivo))
modelo.to(dispositivo)
modelo.eval()
print("Modelo cargado")

# --- VARIABLES COMPARTIDAS ENTRE HILOS ---
frame_actual = None
detecciones_actuales = []
hilo_ejecutandose = True
lock = threading.Lock()


# --- HILO DE DETECCIÓN ---
def hilo_deteccion():
    global frame_actual, detecciones_actuales, hilo_ejecutandose

    while hilo_ejecutandose:
        # Esperar a que haya un frame nuevo
        if frame_actual is None:
            time.sleep(0.005)
            continue

        # Copiar el frame actual para procesar
        with lock:
            frame_a_procesar = frame_actual.copy()

        # Reducir resolución para la detección (más rápido)
        altura, ancho = frame_a_procesar.shape[:2]
        escala = 0.5  # Reducir a la mitad para procesar más rápido
        frame_reducido = cv2.resize(frame_a_procesar, (int(ancho * escala), int(altura * escala)))

        # Convertir a tensor
        frame_rgb = cv2.cvtColor(frame_reducido, cv2.COLOR_BGR2RGB)
        imagen_pil = Image.fromarray(frame_rgb)
        imagen_tensor = F.to_tensor(imagen_pil).unsqueeze(0).to(dispositivo)

        # Detección
        with torch.no_grad():
            prediccion = modelo(imagen_tensor)[0]

        # Escalar coordenadas de vuelta al tamaño original
        cajas = prediccion["boxes"].cpu() / escala
        etiquetas = prediccion["labels"].cpu()
        puntuaciones = prediccion["scores"].cpu()

        # Guardar resultados
        with lock:
            detecciones_actuales = list(zip(cajas, etiquetas, puntuaciones))

        # Pequeña pausa para no saturar la CPU
        time.sleep(0.03)  # 30 FPS de procesamiento


# Iniciar hilo de detección
hilo_detector = threading.Thread(target=hilo_deteccion, daemon=True)
hilo_detector.start()

# --- CAPTURA DE CÁMARA (HILO PRINCIPAL) ---
camara = cv2.VideoCapture(0)

# Optimizar la cámara para mejor FPS
camara.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camara.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
camara.set(cv2.CAP_PROP_FPS, 30)

if not camara.isOpened():
    print("No se pudo acceder a la cámara.")
    hilo_ejecutandose = False
    exit()

print("--------------------------------------------------")
print("Presiona la tecla 'q' en la ventana del video para salir.")
print("--------------------------------------------------")

# Variables para medir FPS
fps_mostrado = 0
ultimo_tiempo = time.time()
contador_frames = 0

while True:
    exito, fotograma = camara.read()
    if not exito:
        print("Se perdió la señal de la cámara.")
        break

    # Actualizar el frame para el hilo de detección
    with lock:
        frame_actual = fotograma.copy()

    # Obtener las últimas detecciones
    with lock:
        detecciones_para_dibujar = detecciones_actuales.copy()

    # Dibujar detecciones
    for caja, etiqueta, puntuacion in detecciones_para_dibujar:
        if puntuacion.item() >= Umbral_confianza:
            id_etiqueta = int(etiqueta.item())

            if id_etiqueta in Indice_a_clase:
                nombre_clase = Indice_a_clase[id_etiqueta]
                porcentaje_seguridad = puntuacion.item() * 100
                xmin, ymin, xmax, ymax = map(int, caja.tolist())

                # Limitar coordenadas dentro del frame
                xmin = max(0, xmin)
                ymin = max(0, ymin)
                xmax = min(fotograma.shape[1], xmax)
                ymax = min(fotograma.shape[0], ymax)

                # Dibujar rectángulo
                cv2.rectangle(fotograma, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)

                # Dibujar texto
                texto_pantalla = f"{nombre_clase}: {porcentaje_seguridad:.1f}%"
                cv2.putText(
                    fotograma, texto_pantalla, (xmin, max(ymin - 10, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
                )

    # Calcular y mostrar FPS
    contador_frames += 1
    tiempo_actual = time.time()
    if tiempo_actual - ultimo_tiempo >= 1.0:
        fps_mostrado = contador_frames
        contador_frames = 0
        ultimo_tiempo = tiempo_actual

    # Mostrar FPS en la esquina superior izquierda
    cv2.putText(fotograma, f"FPS: {fps_mostrado}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("Detector de Camisas", fotograma)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Cerrando el programa")
        break

# Limpieza
hilo_ejecutandose = False
camara.release()
cv2.destroyAllWindows()